# Exploratory data analysis and validation

This notebook explores and validates the preprocessed cholera outbreaks and ECVs sea surface salinity, chlorophyll-a concentration and land surface temperature.

## Setup

In [ ]:
# import packages
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

## Cholera outbreaks

Load cholera outbreaks and explore them briefly.

In [ ]:
cholera_outbreaks = gpd.read_file(
    "../../data/outbreaks/monthly_cholera_outbreaks_india_district_2010_2018.shp"
)
cholera_outbreaks.shape

In [ ]:
cholera_outbreaks.crs

In [ ]:
cholera_outbreaks.info()

In [ ]:
cholera_outbreaks.head()

In [ ]:
# number of states
cholera_outbreaks["state"].nunique()

In [ ]:
# number of districts
cholera_outbreaks["district"].nunique()

In [ ]:
# number of cholera outbreaks
cholera_outbreaks["outbreak"].sum()

In [ ]:
# cholera outbreaks per year
cholera_outbreaks[["year", "outbreak"]].groupby("year").sum().plot.bar(
    title="Cholera outbreaks in India per year (2010 to 2018)", legend=0
)
plt.show()

### Check state-district relationship

In [ ]:
cholera_outbreaks[["state", "district"]].shape

In [ ]:
cholera_outbreaks[["state", "district"]].drop_duplicates().shape

In [ ]:
# districts per state
districts_per_state = (
    cholera_outbreaks[["state", "district"]].groupby("state").agg("nunique").reset_index()
)
districts_per_state

In [ ]:
# states per district
states_per_district = (
    cholera_outbreaks[["state", "district"]].groupby("district").agg("nunique").reset_index()
)
states_per_district

Are there any districts with more than one state?

In [ ]:
states_per_district[states_per_district["state"] != 1]

### Validate cholera outbreaks

The number of cholera outbreaks per state for 2011 to 2015 is available at the [IDSP](https://www.idsp.nic.in/index1.php?lang=1&level=1&sublinkid=5803&lid=3751). The following table has been created from these reports.

In [ ]:
validation_data = pd.read_csv(
    "../../data/outbreaks/cholera_outbreaks_india_validation_2011_2015.csv"
)
validation_data.shape

In [ ]:
validation_data.info()

In [ ]:
validation_data.head()

In [ ]:
validation_data.tail(1)

#### Total

In [ ]:
cholera_outbreaks_total_true = validation_data.tail(1).T.iloc[1:, :].reset_index()
cholera_outbreaks_total_true.columns = ["year", "cholera_outbreaks_true"]
cholera_outbreaks_total_true

In [ ]:
cholera_outbreaks_total_extracted = (
    cholera_outbreaks.loc[
        (cholera_outbreaks["year"] >= 2011) & (cholera_outbreaks["year"] <= 2015)
    ][["year", "outbreak"]]
    .groupby("year")
    .sum()
    .reset_index()
)
cholera_outbreaks_total_extracted = cholera_outbreaks_total_extracted.rename(
    columns={"outbreak": "cholera_outbreaks_extracted"}
)
cholera_outbreaks_total_extracted

In [ ]:
cholera_outbreaks_total_all = pd.concat(
    [cholera_outbreaks_total_true, cholera_outbreaks_total_extracted.drop("year", axis=1)], axis=1
)
cholera_outbreaks_total_all["delta_absolute"] = (
    cholera_outbreaks_total_all["cholera_outbreaks_true"]
    - cholera_outbreaks_total_all["cholera_outbreaks_extracted"]
)
cholera_outbreaks_total_all["delta_relative"] = (
    cholera_outbreaks_total_all["delta_absolute"]
    / cholera_outbreaks_total_all["cholera_outbreaks_true"]
)
cholera_outbreaks_total_all

In [ ]:
round(cholera_outbreaks_total_all["delta_relative"].abs().mean(), 2)

#### State-level

In [ ]:
cholera_outbreaks_state_true = pd.melt(
    validation_data.head(-1), id_vars="State", value_vars=["2011", "2012", "2013", "2014", "2015"]
)
cholera_outbreaks_state_true.columns = ["state", "year", "cholera_outbreaks_true"]
cholera_outbreaks_state_true["state"] = cholera_outbreaks_state_true["state"].str.lower()
cholera_outbreaks_state_true["state"] = cholera_outbreaks_state_true["state"].str.strip()
cholera_outbreaks_state_true["state"] = cholera_outbreaks_state_true["state"].str.replace(
    "&", "and"
)
cholera_outbreaks_state_true.loc[cholera_outbreaks_state_true["state"] == "delhi", "state"] = (
    "nct of delhi"
)
cholera_outbreaks_state_true

In [ ]:
cholera_outbreaks_state_true["state"].nunique()

In [ ]:
cholera_outbreaks_state_extracted = (
    cholera_outbreaks.loc[
        (cholera_outbreaks["year"] >= 2011) & (cholera_outbreaks["year"] <= 2015)
    ][["state", "year", "outbreak"]]
    .groupby(["state", "year"])
    .sum()
    .reset_index()
)
cholera_outbreaks_state_extracted.columns = ["state", "year", "cholera_outbreaks_extracted"]
cholera_outbreaks_state_extracted["year"] = cholera_outbreaks_state_extracted["year"].astype(str)
cholera_outbreaks_state_extracted

In [ ]:
cholera_outbreaks_state_extracted["state"].nunique()

In [ ]:
cholera_outbreaks_state_all = pd.merge(
    cholera_outbreaks_state_true,
    cholera_outbreaks_state_extracted,
    how="left",
    on=["state", "year"],
)
cholera_outbreaks_state_all = cholera_outbreaks_state_all.fillna(0)
cholera_outbreaks_state_all["delta_absolute"] = (
    cholera_outbreaks_state_all["cholera_outbreaks_true"]
    - cholera_outbreaks_state_all["cholera_outbreaks_extracted"]
)
cholera_outbreaks_state_all["delta_relative"] = (
    cholera_outbreaks_state_all["delta_absolute"]
    / cholera_outbreaks_state_all["cholera_outbreaks_true"]
)
cholera_outbreaks_state_all = cholera_outbreaks_state_all.fillna(0)
cholera_outbreaks_state_all

In [ ]:
# true == extracted
cholera_outbreaks_state_all[
    cholera_outbreaks_state_all["cholera_outbreaks_true"]
    == cholera_outbreaks_state_all["cholera_outbreaks_extracted"]
].groupby("year").size()

In [ ]:
# true != extracted
cholera_outbreaks_state_all[
    cholera_outbreaks_state_all["cholera_outbreaks_true"]
    != cholera_outbreaks_state_all["cholera_outbreaks_extracted"]
].groupby("year").size()

Unfortunately, not all cholera outbreaks seem to have been correctly identified for 2011 to 2015. Most likely not all cholera outbreaks have been correctly identified for 2010 and 2016 to 2018 as well.

### Create static map

First, we need to aggregate the cholera outbreaks on state and district level.

In [ ]:
# aggregate cholera outbreaks by state and district
map_data = (
    cholera_outbreaks[["state", "district", "outbreak"]]
    .groupby(["state", "district"])
    .sum()
    .reset_index()
)
map_data = pd.merge(
    cholera_outbreaks[["state", "district", "geometry"]].drop_duplicates(),
    map_data,
    how="inner",
    on=["state", "district"],
)
map_data

Then we need to get the geometries of states and districts which are available at the [Database of Global Administrative Areas](https://www.gadm.org/index.html) and have already been downloaded during preprocessing the cholera outbreaks.

In [ ]:
india = gpd.read_file("../../data/outbreaks/gadm36_IND_shp/gadm36_IND_2.shp")
india.shape

In [ ]:
india.crs

In [ ]:
india.info()

In [ ]:
india.head()

In [ ]:
india.plot()

In [ ]:
# state polygons
states = india[["NAME_1", "geometry"]].dissolve(by="NAME_1")

In [ ]:
states.plot()

In [ ]:
# district polygons
districts = india[["NAME_2", "geometry"]]

In [ ]:
districts.plot()

In [ ]:
# check whether crs are identical
india.crs == map_data.crs

In [ ]:
%%time

# create plot and set figure size
fig, ax = plt.subplots(figsize=(14, 10))

# plot district boundaries
districts.plot(ax=ax, facecolor="lightgray", edgecolor="black", linewidth=0.5, alpha=0.7)

# plot state boundaries
states.plot(ax=ax, facecolor="white", edgecolor="black", linewidth=1.5, alpha=0.8)

# plot cholera outbreaks
map_data.plot(
    ax=ax,
    column="outbreak",
    cmap="YlOrRd",
    edgecolor="black",
    linewidth=0.5,
    legend=True,
    legend_kwds={"label": "Number of cholera outbreaks"},
)

# set plot parameters
ax.set_title(
    "Cholera outbreaks in India by district (2010 to 2018)",
    fontdict={"fontsize": "25", "fontweight": "3"},
)
ax.set_xlabel("Longitude (°E)", fontdict={"fontsize": "16", "fontweight": "3"})
ax.set_ylabel("Latitude (°N)", fontdict={"fontsize": "16", "fontweight": "3"})
plt.tick_params(labeltop=True, labelright=True)
plt.grid(alpha=0.5)
plt.tight_layout()

# save plot
plt.savefig("../0_images/cholera_outbreaks_india_district_2010_2018.png", dpi=300)

## ECVs

In [ ]:
def plot_ecv(df, ecv_short, ecv_long):
    # create subplots
    fig, axs = plt.subplots(2, 2, figsize=(8, 8))

    # set min and max values for x and y axses
    x_min = 60
    x_max = 100
    y_min = 0
    y_max = 40

    # set x and y labels
    x_label = "Longitude (°E)"
    y_label = "Latitude (°N)"

    # set alpha
    alpha = 0.5

    # plot data for January
    df[df["month"] == 1].plot(column=ecv_short, legend=True, ax=axs[0, 0])
    axs[0, 0].set_xlim(x_min, x_max)
    axs[0, 0].set_ylim(y_min, y_max)
    axs[0, 0].set_title("January")
    axs[0, 0].set_xlabel(x_label)
    axs[0, 0].set_ylabel(y_label)
    axs[0, 0].grid(alpha=alpha)

    # plot data for April
    df[df["month"] == 4].plot(column=ecv_short, legend=True, ax=axs[0, 1])
    axs[0, 1].set_xlim(x_min, x_max)
    axs[0, 1].set_ylim(y_min, y_max)
    axs[0, 1].set_title("April")
    axs[0, 1].set_xlabel(x_label)
    axs[0, 1].set_ylabel(y_label)
    axs[0, 1].grid(alpha=alpha)

    # plot data for July
    df[df["month"] == 7].plot(column=ecv_short, legend=True, ax=axs[1, 0])
    axs[1, 0].set_xlim(x_min, x_max)
    axs[1, 0].set_ylim(y_min, y_max)
    axs[1, 0].set_title("July")
    axs[1, 0].set_xlabel(x_label)
    axs[1, 0].set_ylabel(y_label)
    axs[1, 0].grid(alpha=alpha)

    # plot data for October
    df[df["month"] == 10].plot(column=ecv_short, legend=True, ax=axs[1, 1])
    axs[1, 1].set_xlim(x_min, x_max)
    axs[1, 1].set_ylim(y_min, y_max)
    axs[1, 1].set_title("October")
    axs[1, 1].set_xlabel(x_label)
    axs[1, 1].set_ylabel(y_label)
    axs[1, 1].grid(alpha=alpha)

    # set figure title
    year = df["year"].unique()[0]
    plt.suptitle(f"Mean {ecv_long} {year}", fontsize=16)

    plt.tight_layout()
    plt.show()

    # save plot
    fig.savefig(f"../0_images/mean_{ecv_short}_{year}.png", dpi=300)

### Sea surface salinity

In [ ]:
path = "../../data/sea_surface_salinity/"
ecv_short = "sss"
year = 2018

In [ ]:
%%time

sss = gpd.read_file(os.path.join(path, f"monthly_{ecv_short}_{year}.shp"))
sss.shape

In [ ]:
sss.info()

In [ ]:
sss.head()

In [ ]:
%%time

plot_ecv(sss, "sss", "sea surface salinity")

### Chlorophyll-a concentration

In [ ]:
path = "../../data/chlorophyll_a_concentration/"
ecv_short = "chlora"
year = 2018

In [ ]:
%%time

chlora = gpd.read_file(os.path.join(path, f"monthly_{ecv_short}_{year}.shp"))
chlora.shape

In [ ]:
chlora.info()

In [ ]:
chlora.head()

In [ ]:
%%time

plot_ecv(chlora, "chlora", "chlorophyll-a concentration")

### Land surface temperature

In [ ]:
path = "../../data/land_surface_temperature"
ecv_short = "lst"
year = 2018

In [ ]:
%%time

lst = gpd.read_file(os.path.join(path, f"monthly_{ecv_short}_{year}.shp"))
lst.shape

In [ ]:
lst.info()

In [ ]:
lst.head()

In [ ]:
%%time

plot_ecv(lst, "lst", "land surface temperature")